# Structure Preservation

Cleaning removes extraction noise.

Structure preservation asks a different question:

> **What organizational information can we preserve from the source document?**

A real document contains relationships between:

```text
Document
 ├── Title
 ├── Section
 │    ├── Subsection
 │    └── Paragraph
 └── List
```

Those relationships can become useful context for later metadata, chunking, retrieval, and citations.

In this notebook we use a **real machine-readable PDF asset** and inspect its actual text, font sizes, font styles, and positions before constructing a structured representation.


## Learning Objectives

By the end of this notebook, you should be able to:

- Explain why structure matters for RAG.
- Inspect structural evidence from a PDF.
- Use font size, font style, position, and text patterns as signals.
- Distinguish headings from ordinary body text.
- Preserve paragraphs and lists.
- Detect numbered sections and subsections.
- Build a hierarchical document representation.
- Understand reading-order problems.
- Validate preserved structure.
- Explain why structure preservation should precede chunking.

## 1. The Problem We Are Solving

A flat extraction might look like:

```text
Employee Travel Policy
JoTeq the First
1. Purpose
This policy explains...
2. Eligibility
Employees travelling...
3. Reimbursement
3.1 Accommodation
Hotel expenses...
```

The words are present, but their relationships are implicit.

We want something closer to:

```text
Employee Travel Policy
│
├── 1. Purpose
│   └── Paragraph
│
├── 2. Eligibility
│   ├── Paragraph
│   └── Paragraph
│
└── 3. Reimbursement
    ├── 3.1 Accommodation
    │   └── Paragraph
    └── 3.2 Transportation
        └── Paragraph
```

The key difference is that the second representation preserves organization.

## 2. Use a Real Machine-Readable Document

This notebook deliberately uses a **text PDF**, not the scanned OCR asset.

Why?

Because this notebook is about preserving structure from parser output.

The OCR notebook already covers:

```text
Image / scanned PDF
        ↓
       OCR
        ↓
      Text
```

Here we want:

```text
Machine-readable PDF
        ↓
      Parsing
        ↓
Text + layout evidence
        ↓
Structure preservation
```

## 3. Install and Import

In [1]:
!pip install -q pymupdf


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path
import re
import json
from collections import Counter

import pymupdf

## 4. Load the Document

The PDF is stored in the course `data/` directory.

In [3]:
DATA_DIR = Path("data")
PDF_PATH = DATA_DIR / "travel_policy.pdf"

if not PDF_PATH.exists():
    raise FileNotFoundError(f"Missing asset: {PDF_PATH}")

print(PDF_PATH)


data/travel_policy.pdf


## 5. Inspect the Raw Text First

Before designing structure rules, inspect what the parser actually returns.

In [4]:
with pymupdf.open(PDF_PATH) as document:
    for page_number, page in enumerate(document, start=1):
        print(f"--- PAGE {page_number} ---")
        print(page.get_text("text"))

--- PAGE 1 ---
Employee Travel Policy
JoTeq the First
1. Purpose
This policy explains how employees should request and claim approved business travel expenses.
2. Eligibility
Employees travelling for approved company business may claim reasonable and necessary expenses.
Travel must be approved before the journey begins unless an emergency exception has been granted.
3. Reimbursement
3.1 Accommodation
Hotel expenses are reimbursable within the approved limits. Employees should provide an itemized
receipt.
3.2 Transportation
Airfare, approved local transportation, and other eligible travel costs may be reimbursed.

--- PAGE 2 ---
Employee Travel Policy · Continued
JoTeq the First
4. Meals
Employees may claim eligible business-travel meals within the applicable daily allowance.
Alcoholic beverages are not reimbursable.
5. Required Documentation
- Approved travel request
- Receipts for reimbursable expenses
- Completed expense claim
6. Submission Deadline
Expense claims should be submitted

Notice that we have not told the code what the headings are.

This is important.

We first observe the extracted representation and then ask what evidence can distinguish:

- title
- heading
- subsection
- paragraph
- list

## 6. Inspect Layout and Font Evidence

PDF text spans contain more information than just their characters.

We can inspect:

- text
- font
- font size
- bounding box
- block and line relationships

These are useful structural signals.

In [5]:
with pymupdf.open(PDF_PATH) as document:
    for page_number, page in enumerate(document, start=1):
        print(f"--- PAGE {page_number} ---")

        page_dict = page.get_text("dict")
        
        for block in page_dict["blocks"]:
            if block["type"] != 0:
                continue

            for line in block["lines"]:
                for span in line["spans"]:
                    print({
                        "text": span["text"].strip(),
                        "font": span["font"],
                        "size": round(span["size"], 1),
                        "bbox": tuple(round(v, 1) for v in span["bbox"]),
                    })

--- PAGE 1 ---
{'text': 'Employee Travel Policy', 'font': 'Helvetica-Bold', 'size': 22.0, 'bbox': (50.0, 31.5, 295.8, 61.8)}
{'text': 'JoTeq the First', 'font': 'Helvetica-Bold', 'size': 10.0, 'bbox': (50.0, 76.3, 121.7, 90.1)}
{'text': '1. Purpose', 'font': 'Helvetica-Bold', 'size': 15.0, 'bbox': (50.0, 98.9, 126.7, 119.6)}
{'text': 'This policy explains how employees should request and claim approved business travel expenses.', 'font': 'Helvetica', 'size': 10.5, 'bbox': (55.0, 138.0, 511.9, 152.4)}
{'text': '2. Eligibility', 'font': 'Helvetica-Bold', 'size': 15.0, 'bbox': (50.0, 181.9, 133.4, 202.6)}
{'text': 'Employees travelling for approved company business may claim reasonable and necessary expenses.', 'font': 'Helvetica', 'size': 10.5, 'bbox': (55.0, 221.0, 530.6, 235.4)}
{'text': 'Travel must be approved before the journey begins unless an emergency exception has been granted.', 'font': 'Helvetica', 'size': 10.5, 'bbox': (55.0, 269.0, 530.7, 283.4)}
{'text': '3. Reimbursement',

Now we have actual evidence.

For example, the title and headings use a larger/bolder font than normal body text.

That means we can use **layout and typography as signals**, rather than hard-coding a list of headings.

## 7. Group Spans into Lines

Structural decisions are usually easier at the line level than at the individual span level.

We can extract lines while retaining their strongest layout information.

In [6]:
def extract_lines(pdf_path):
    records = []

    with pymupdf.open(pdf_path) as document:
        for page_number, page in enumerate(document, start=1):
            page_dict = page.get_text("dict")

            for block in page_dict["blocks"]:
                if block["type"] != 0:
                    continue

                for line in block["lines"]:
                    spans = line["spans"]
                    line_text = "".join(span["text"] for span in spans).strip()

                    if not line_text:
                        continue
                    
                    records.append({
                        "page": page_number,
                        "text": line_text,
                        "bbox": line["bbox"],
                        "max_font_size": max(span["size"] for span in spans),
                        "fonts": [span["font"] for span in spans],
                    })
    return records

lines = extract_lines(PDF_PATH)

for record in lines:
    print(record)

{'page': 1, 'text': 'Employee Travel Policy', 'bbox': (50.0, 31.459999084472656, 295.783935546875, 61.75400161743164), 'max_font_size': 22.0, 'fonts': ['Helvetica-Bold']}
{'page': 1, 'text': 'JoTeq the First', 'bbox': (50.0, 76.30000305175781, 121.67999267578125, 90.06999969482422), 'max_font_size': 10.0, 'fonts': ['Helvetica-Bold']}
{'page': 1, 'text': '1. Purpose', 'bbox': (50.0, 98.94999694824219, 126.69500732421875, 119.6050033569336), 'max_font_size': 15.0, 'fonts': ['Helvetica-Bold']}
{'page': 1, 'text': 'This policy explains how employees should request and claim approved business travel expenses.', 'bbox': (55.0, 137.99996948242188, 511.93927001953125, 152.42697143554688), 'max_font_size': 10.5, 'fonts': ['Helvetica']}
{'page': 1, 'text': '2. Eligibility', 'bbox': (50.0, 181.9499969482422, 133.3699951171875, 202.60499572753906), 'max_font_size': 15.0, 'fonts': ['Helvetica-Bold']}
{'page': 1, 'text': 'Employees travelling for approved company business may claim reasonable and ne

## 8. Discover Typography Patterns

Instead of assuming a heading size, inspect the sizes that actually occur.

In [7]:
font_sizes = Counter(
    round(record["max_font_size"], 1)
    for record in lines
)

font_sizes

Counter({10.5: 14, 15.0: 6, 22.0: 2, 10.0: 2})

This gives us an empirical view of the document.

A document might contain:

```text
22 pt → title
15 pt → headings
10.5 pt → body
```

Those values are properties of this particular document.

A production parser should avoid assuming that every PDF uses these exact sizes.

## 9. Identify Candidate Headings from Evidence

For this document, a candidate heading has several signals:

- relatively large font size
- bold font
- short line
- numbered-section pattern or heading-like wording

We can combine signals instead of relying on one arbitrary rule.

In [8]:
SECTION_PATTERN = re.compile(
    r"^\s*(\d+(?:\.\d+)*)\.?\s+(.+)$"
)

def is_bold_font(fonts):
    return any(
        "bold" in font.lower() or "hebo" in font.lower()
        for font in fonts
    )

def heading_score(record):
    text = record["text"]
    score = 0

    if record["max_font_size"] >= 15:
        score += 2

    if is_bold_font(record["fonts"]):
        score += 1

    if len(text) <= 80:
        score += 1

    if SECTION_PATTERN.match(text):
        score += 2

    return score

candidates = [
    (record["text"], heading_score(record))
    for record in lines
]

for text, score in candidates:
    if score >= 3:
        print(score, repr(text))

4 'Employee Travel Policy'
6 '1. Purpose'
6 '2. Eligibility'
6 '3. Reimbursement'
3 '3.1 Accommodation'
3 '3.2 Transportation'
4 'Employee Travel Policy · Continued'
6 '4. Meals'
6 '5. Required Documentation'
6 '6. Submission Deadline'


This is fundamentally different from:

```python
known_headings = {...}
```

We are not giving the system the answer.

We are using observable document properties as evidence.

## 10. Distinguish Titles, Headings, and Subsections

A structural parser may classify the largest heading-like element as the title.

Numbered headings can then be assigned hierarchy based on their numbering.

For example:

```text
3. Reimbursement
3.1 Accommodation
3.2 Transportation
```

can be represented as:

```text
3
├── 3.1
└── 3.2
```

In [9]:
def section_number(text):
    match = SECTION_PATTERN.match(text)
    return match.group(1) if match else None

for record in lines:
    number = section_number(record["text"])
    if number:
        print(number, record["text"])

1 1. Purpose
2 2. Eligibility
3 3. Reimbursement
3.1 3.1 Accommodation
3.2 3.2 Transportation
4 4. Meals
5 5. Required Documentation
6 6. Submission Deadline


The numbering gives us an additional structural signal.

It is not the only signal: layout, typography, and position still matter.

## 11. Preserve Paragraphs

Body text often appears as consecutive lines within a paragraph.

We should combine those lines without accidentally combining separate sections.

A simple paragraph grouping strategy can use:

- same page
- compatible vertical position
- similar body font
- absence of a new heading
- spacing between blocks

For this exercise, we will first identify heading boundaries.

In [10]:
heading_records = [
    record
    for record in lines
    if heading_score(record) >= 3
]

for record in heading_records:
    print(
        f"PAGE {record['page']} | "
        f"{record['text']} | "
        f"score={heading_score(record)}"
    )

PAGE 1 | Employee Travel Policy | score=4
PAGE 1 | 1. Purpose | score=6
PAGE 1 | 2. Eligibility | score=6
PAGE 1 | 3. Reimbursement | score=6
PAGE 1 | 3.1 Accommodation | score=3
PAGE 1 | 3.2 Transportation | score=3
PAGE 2 | Employee Travel Policy · Continued | score=4
PAGE 2 | 4. Meals | score=6
PAGE 2 | 5. Required Documentation | score=6
PAGE 2 | 6. Submission Deadline | score=6


The next step is to assign body lines to the current section.

This is where structure becomes a document hierarchy rather than a collection of isolated lines.

## 12. Build a Section-Aware Representation

We will create sections from the detected numbered headings.

This is still a simplified educational parser. The important part is that the structure is derived from the document evidence.

In [11]:
def build_sections(records):
    sections = []
    current = None

    for record in records:
        text = record["text"]

        if heading_score(record) >= 3:
            number = section_number(text)

            if current is not None:
                sections.append(current)

            current = {
                "number": number,
                "heading": text,
                "page": record["page"],
                "content": [],
            }

        elif current is not None:
            current["content"].append(text)

    if current is not None:
        sections.append(current)

    return sections

sections = build_sections(lines)

print(json.dumps(sections, indent=2, ensure_ascii=False))

[
  {
    "number": null,
    "heading": "Employee Travel Policy",
    "page": 1,
    "content": [
      "JoTeq the First"
    ]
  },
  {
    "number": "1",
    "heading": "1. Purpose",
    "page": 1,
    "content": [
      "This policy explains how employees should request and claim approved business travel expenses."
    ]
  },
  {
    "number": "2",
    "heading": "2. Eligibility",
    "page": 1,
    "content": [
      "Employees travelling for approved company business may claim reasonable and necessary expenses.",
      "Travel must be approved before the journey begins unless an emergency exception has been granted."
    ]
  },
  {
    "number": "3",
    "heading": "3. Reimbursement",
    "page": 1,
    "content": []
  },
  {
    "number": "3.1",
    "heading": "3.1 Accommodation",
    "page": 1,
    "content": [
      "Hotel expenses are reimbursable within the approved limits. Employees should provide an itemized",
      "receipt."
    ]
  },
  {
    "number": "3.2",
    "headi

We have now moved from:

```text
PDF → flat lines
```

to:

```text
PDF
 ↓
lines + layout evidence
 ↓
candidate structural elements
 ↓
sections
```

## 13. Build Hierarchy from Section Numbers

Now use numbering to establish parent-child relationships.

For example:

```text
3
3.1
3.2
```

means:

```text
3
├── 3.1
└── 3.2
```

In [12]:
def section_level(number):
    if not number:
        return None
    return len(number.split("."))

for section in sections:
    print(
        section["number"],
        "level=",
        section_level(section["number"]),
        section["heading"]
    )

None level= None Employee Travel Policy
1 level= 1 1. Purpose
2 level= 1 2. Eligibility
3 level= 1 3. Reimbursement
3.1 level= 2 3.1 Accommodation
3.2 level= 2 3.2 Transportation
None level= None Employee Travel Policy · Continued
4 level= 1 4. Meals
5 level= 1 5. Required Documentation
6 level= 1 6. Submission Deadline


The hierarchy can be inferred from the section-number pattern.

This is stronger than assuming that every indented line is a subsection.

## 14. Preserve Lists

The second page contains a list of required documents.

A list item can be detected using its marker:

```text
- Approved travel request
- Receipts
- Completed expense claim
```

But we preserve the list as a list rather than flattening it into an ordinary paragraph.

In [13]:
LIST_PATTERN = re.compile(r"^\s*[-*+]\s+(.+)$")

for record in lines:
    match = LIST_PATTERN.match(record["text"])

    if match:
        print("LIST ITEM:", match.group(1))

LIST ITEM: Approved travel request
LIST ITEM: Receipts for reimbursable expenses
LIST ITEM: Completed expense claim


In more complex documents, list detection can also use indentation, numbering, layout, and neighboring lines.

## 15. Reading Order

Structure preservation also means preserving the intended reading order.

A page can contain:

```text
Column A          Column B

Paragraph 1       Sidebar
Paragraph 2       Note
Paragraph 3       Warning
```

A naive extraction order may interleave the columns.

For complex layouts, inspect:

- x/y coordinates
- block boundaries
- columns
- reading direction
- parser output

Do not assume that the order returned by a parser is always the human reading order.

## 16. Preserve Provenance During Structure Building

Even though metadata and provenance are the next notebook, structure records should not throw away source information.

For example:

```python
{
    "heading": "2. Eligibility",
    "page": 1,
    "content": [...]
}
```

The page number is already useful provenance.

The next notebook will formalize this into a broader metadata model.

## 17. Validate the Structure

A structure parser can fail silently.

We should check:

- Is a title present?
- Were expected numbered sections detected?
- Are section numbers in sensible order?
- Are subsections attached to the right parent?
- Are list items preserved?
- Does every content block belong somewhere?

In [14]:
detected_numbers = [
    section["number"]
    for section in sections
    if section["number"]
]

validation = {
    "section_count": len(sections),
    "detected_section_numbers": detected_numbers,
    "has_numbered_sections": bool(detected_numbers),
    "has_title_candidate": any(
        record["max_font_size"] >= 20
        for record in lines
    ),
}

validation

{'section_count': 10,
 'detected_section_numbers': ['1', '2', '3', '3.1', '3.2', '4', '5', '6'],
 'has_numbered_sections': True,
 'has_title_candidate': True}

Validation gives us a checkpoint before the document moves to metadata, chunking, and retrieval.

## 18. Why Not Just Flatten Everything?

A flat representation loses relationships.

Instead of:

```text
Hotel expenses are reimbursable...
```

we can preserve:

```python
{
    "text": "Hotel expenses are reimbursable...",
    "section": "3.1 Accommodation",
    "parent_section": "3. Reimbursement",
    "page": 1,
}
```

The same sentence now carries context.

## 19. Structure Preservation and Chunking

We are still **not chunking**.

But structure gives the future chunker better boundaries.

A chunk can later inherit:

```text
Document title
→ Section
→ Subsection
→ Paragraph
```

instead of containing an isolated fragment with no context.

## 20. What This Approach Does Not Solve

This notebook uses layout and textual signals to demonstrate the engineering idea.

Real documents can be much harder:

- multi-column reports
- inconsistent heading styles
- complex forms
- nested tables
- scanned pages
- figures and captions
- visually positioned labels

Those cases may require specialized document parsers or advanced document-understanding models.

The principle remains:

> **Use evidence from the document rather than assuming a universal structure.**


## 21. Exercise

Using the provided travel-policy PDF:

1. Inspect the raw extracted text.
2. Inspect font sizes and positions.
3. Identify the title using document evidence.
4. Identify section candidates without hard-coding their names.
5. Detect numbered sections and subsections.
6. Preserve paragraphs.
7. Detect the required-documents list.
8. Build a hierarchical representation.
9. Preserve page numbers with structural elements.
10. Validate the resulting structure.
11. Explain which structural decisions are reliable and which remain heuristic.
12. Explain how this representation could improve future chunking.

## Key Takeaways

1. Structure preservation is different from cleaning.
2. Do not hard-code the document's headings when teaching structure detection.
3. PDF layout and typography provide useful structural evidence.
4. Font size, font style, position, numbering, and text patterns can be combined as signals.
5. Preserve paragraph and list boundaries.
6. Section numbering can reveal hierarchy.
7. Reading order is an important structural problem.
8. Keep provenance information available while building structure.
9. Validate structural extraction before downstream processing.
10. Structure-aware representations give chunking and retrieval more context.
11. Complex documents may require specialized parsers or advanced document understanding.

## What's Next?

We now have a structured representation:

```text
Document
 ├── Title
 ├── Sections
 │    ├── Subsections
 │    └── Content
 └── Lists
```

But we still need to know:

> **Exactly where did each piece of content come from, and which version of the document produced it?**

Next:

## Metadata & Provenance

We will formalize source identifiers, pages, sections, timestamps, versions, and extraction information so that content remains traceable throughout the RAG pipeline.